In [1]:
from langchain_community.document_loaders import TextLoader


In [6]:
loader=TextLoader('ddata.txt',encoding='utf-8')

In [7]:
docs=loader.load()
docs

[Document(metadata={'source': 'ddata.txt'}, page_content='\n\n\nDiabetes\nDiabetes is a common condition that affects people of all ages. There are several forms of diabetes. Type 2 is the most common. A combination of treatment strategies can help you manage the condition to live a healthy life and prevent complications.\n\nContents\nOverview\nSymptoms and Causes\nDiagnosis and Tests\nManagement and Treatment\nOutlook / Prognosis\nPrevention\nLiving With\n\nSymptoms and Causes\nOverview\n\n\nLearn about the early signs of diabetes.\nWhat is diabetes?\nDiabetes is a condition that happens when your blood sugar (glucose) is too high. It develops when your pancreas doesn’t make enough insulin or any at all, or when your body isn’t responding to the effects of insulin properly. Diabetes affects people of all ages. Most forms of diabetes are chronic (lifelong), and all forms are manageable with medications and/or lifestyle changes.\n\nAdvertisement\n\nCleveland Clinic is a non-profit acade

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk_documents=text_splitter.split_documents(docs)

In [9]:
chunk_documents

[Document(metadata={'source': 'ddata.txt'}, page_content='Diabetes\nDiabetes is a common condition that affects people of all ages. There are several forms of diabetes. Type 2 is the most common. A combination of treatment strategies can help you manage the condition to live a healthy life and prevent complications.\n\nContents\nOverview\nSymptoms and Causes\nDiagnosis and Tests\nManagement and Treatment\nOutlook / Prognosis\nPrevention\nLiving With\n\nSymptoms and Causes\nOverview\n\n\nLearn about the early signs of diabetes.\nWhat is diabetes?\nDiabetes is a condition that happens when your blood sugar (glucose) is too high. It develops when your pancreas doesn’t make enough insulin or any at all, or when your body isn’t responding to the effects of insulin properly. Diabetes affects people of all ages. Most forms of diabetes are chronic (lifelong), and all forms are manageable with medications and/or lifestyle changes.\n\nAdvertisement'),
 Document(metadata={'source': 'ddata.txt'}, 

In [10]:
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from typing import List,Tuple,Dict,Set,Any
import numpy as np
import os
import uuid

In [11]:
from sentence_transformers import SentenceTransformer

c:\Users\krish sharma\mini_proj_sem5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from src.exception import CustomException

In [13]:
class embeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
           
        Args:
           model_name:Hugging model name for sentence embeddings

        """

        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            """Load the sentence model transformer"""
            print(f"loading sentence model {self.model_name}")
            self.model=SentenceTransformer(self.model_name)

            print(f"model loaded sucessfully {self.model_name} dimension {self.model.get_sentence_embedding_dimension()}")
            
        except Exception as e:
            raise CustomException(e)
            
    def embedding_text(self,text: List[str]) ->np.ndarray:
        """
        This function takes the list as the input and returns the np.array

        """


        if not self.model:
            raise ValueError("Model not loaded")
        embeddings=self.model.encode(text,show_progress_bar=True)   

        return embeddings


embedding=embeddingManager()
embedding
        

loading sentence model all-MiniLM-L6-v2
model loaded sucessfully all-MiniLM-L6-v2 dimension 384


In [15]:
class MyChromaVectorManager:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "cad_notes_collection",
        persist_directory="C:/Users/krish sharma/mini_proj_sem5/ddata/chromadb",
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        import os, uuid
        import numpy as np
        import chromadb
        from typing import List, Any

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.vector_client = None
        self.vector_collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.vector_client = chromadb.PersistentClient(path=self.persist_directory)
            self.vector_collection = self.vector_client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Text document embedding rag"},
            )
            print(f"Vector store initialized collection {self.collection_name}")
            print(f"Existing documents in collection {self.vector_collection.count()}")
        except Exception as e:
            raise Exception(e)

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add the documents and their embeddings to vector store

        Args:
            documents: List of langchain documents
            embeddings: their corresponding embeddings
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the embeddings")
        print(f"Adding {len(documents)} documents to chromadb")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.vector_collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.vector_collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Usage:
vec = MyChromaVectorManager()

vec


Vector store initialized collection cad_notes_collection
Existing documents in collection 0


In [16]:
texts=[docs.page_content for docs in chunk_documents]
embeddings=embedding.embedding_text(texts)

Batches: 100%|██████████| 2/2 [00:03<00:00,  1.58s/it]


In [17]:
texts

['Diabetes\nDiabetes is a common condition that affects people of all ages. There are several forms of diabetes. Type 2 is the most common. A combination of treatment strategies can help you manage the condition to live a healthy life and prevent complications.\n\nContents\nOverview\nSymptoms and Causes\nDiagnosis and Tests\nManagement and Treatment\nOutlook / Prognosis\nPrevention\nLiving With\n\nSymptoms and Causes\nOverview\n\n\nLearn about the early signs of diabetes.\nWhat is diabetes?\nDiabetes is a condition that happens when your blood sugar (glucose) is too high. It develops when your pancreas doesn’t make enough insulin or any at all, or when your body isn’t responding to the effects of insulin properly. Diabetes affects people of all ages. Most forms of diabetes are chronic (lifelong), and all forms are manageable with medications and/or lifestyle changes.\n\nAdvertisement',
 'Advertisement\n\nCleveland Clinic is a non-profit academic medical center. Advertising on our site 

In [18]:
vec.add_documents(chunk_documents,embeddings)

Adding 45 documents to chromadb
Successfully added 45 documents to vector store
Total documents in collection: 45


In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: vec, embedding_manager: embeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[dict[str, Any]]:
            """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold

            Returns:
                List of dictionaries containing retrieved documents and metadata
            """
            print(f"Retrieving documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")

            # Generate query embedding
            query_embedding = self.embedding_manager.embedding_text([query])[0]

            try:
                results = self.vector_store.vector_collection.query(
                    query_embeddings=[query_embedding.tolist()],
                    n_results=top_k
                )

                # Process results
                retrieved_docs = []

                if results['documents'] and results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]

                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        # Convert distance to similarity score (ChromaDB uses cosine distance)
                        similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    print("No documents found")

                return retrieved_docs
    
            except Exception as e:
                raise CustomException(e)            
            
ragretriver=RAGRetriever(vector_store=vec,embedding_manager=embedding)

In [30]:
ragretriver.retrieve("Higher total cholesterol correlates with plaque formation") 

Retrieving documents for query: 'Higher total cholesterol correlates with plaque formation'
Top K: 5, Score threshold: 0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.64it/s]

Retrieved 0 documents (after filtering)


[]

In [31]:
!pip install langchain_groq

In [32]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [69]:
query="""My age is 24 urea is 19.5 and creatinine is 0.61 and my cholestrol is 161 and triglycerides is 150 and hdl is 42 and ldl is 87 and vldl is 32 and non_hdl is 119 and hba1c is 5.6 and fbs is 90 and ppbs is 120 and my weight is 70kg and height is 175cm what should i do to control my diabetes and i am not able not understand all this terms explain me all the terms and tell me what should i do to control my diabetes """

In [37]:
q="Age is 24, sex is 1, chest pain type is 0, resting systolic BP is 120 mmHg, cholesterol is 161 mg/dL, fasting blood sugar flag is 0, resting ECG is 0, max heart rate is 180 bpm, exercise-induced angina is 0, oldpeak is 0.0, and ST slope is 1. i am not able not understand all this terms explain me all the terms and tell me in detail what should i do to control coronary artery disease"

In [38]:
rag_simple(q,ragretriver,llm,top_k=3)

Retrieving documents for query: 'Age is 24, sex is 1, chest pain type is 0, resting systolic BP is 120 mmHg, cholesterol is 161 mg/dL, fasting blood sugar flag is 0, resting ECG is 0, max heart rate is 180 bpm, exercise-induced angina is 0, oldpeak is 0.0, and ST slope is 1. i am not able not understand all this terms explain me all the terms and tell me in detail what should i do to control coronary artery disease'
Top K: 3, Score threshold: 0.0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.04it/s]

Retrieved 1 documents (after filtering)


"Let's break down these medical terms and then talk about what you can do to manage your heart health.\n\n**Understanding the Terms**\n\n* **Age:** Your age is 24, which is generally considered young and puts you at lower risk for coronary artery disease (CAD).\n* **Sex:** Being male (sex = 1) slightly increases your risk of CAD compared to females.\n* **Chest Pain Type:** 0 indicates you didn't report experiencing chest pain at rest.\n* **Resting Systolic BP:** 120 mmHg is a normal blood pressure reading.\n* **Cholesterol:** 161 mg/dL is considered high cholesterol. High cholesterol is a major risk factor for CAD.\n* **Fasting Blood Sugar Flag:** 0 means your fasting blood sugar level is normal.\n* **Resting ECG:** 0 indicates your resting electrocardiogram (ECG) was normal.\n* **Max Heart Rate:** 180 bpm is a good maximum heart rate for your age. It suggests good cardiovascular fitness.\n* **Exercise-Induced Angina:** 0 means you didn't experience chest pain during the exercise test.

In [39]:
"Let's break down these medical terms and then talk about what you can do to manage your heart health.\n\n**Understanding the Terms**\n\n* **Age:** Your age is 24, which is generally considered young and puts you at lower risk for coronary artery disease (CAD).\n* **Sex:** Being male (sex = 1) slightly increases your risk of CAD compared to females.\n* **Chest Pain Type:** 0 indicates you didn't report experiencing chest pain at rest.\n* **Resting Systolic BP:** 120 mmHg is a normal blood pressure reading.\n* **Cholesterol:** 161 mg/dL is considered high cholesterol. High cholesterol is a major risk factor for CAD.\n* **Fasting Blood Sugar Flag:** 0 means your fasting blood sugar level is normal.\n* **Resting ECG:** 0 indicates your resting electrocardiogram (ECG) was normal.\n* **Max Heart Rate:** 180 bpm is a good maximum heart rate for your age. It suggests good cardiovascular fitness.\n* **Exercise-Induced Angina:** 0 means you didn't experience chest pain during the exercise test.\n* **Oldpeak:** 0.0 indicates no significant ST-segment depression at rest on your ECG.\n* **ST Slope:** 1 suggests a normal ST-segment slope on your ECG.\n\n**What This Means**\n\nBased on the information provided, your overall cardiovascular health appears to be good. You have a normal resting blood pressure, a good maximum heart rate, and no signs of ischemia (reduced blood flow to the heart) at rest or during exercise. However, your high cholesterol is a concern and needs to be addressed.\n\n**Controlling Coronary Artery Disease**\n\nWhile you don't currently show signs of CAD, it's important to take steps to prevent it in the future. Here's what you can do:\n\n1. **Lower Your Cholesterol:**\n\n   * **Diet:** Eat a heart-healthy diet low in saturated and trans fats, cholesterol, and sodium. Focus on fruits, vegetables, whole grains, lean protein, and healthy fats.\n   * **Exercise:** Aim for at least 30 minutes of moderate-intensity exercise most days of the week.\n   * **Medication:** Your doctor may prescribe medication to help lower your cholesterol if lifestyle changes alone aren't enough.\n\n2. **Manage Your Blood Pressure:**\n\n   * **Healthy Diet:** Follow the same dietary recommendations as for cholesterol.\n   * **Exercise:** Regular physical activity can help lower blood pressure.\n   * **Limit Salt:** Reduce your sodium intake.\n   * **Medication:** If necessary, your doctor may prescribe medication to control your blood pressure.\n\n3. **Maintain a Healthy Weight:**\n\n   * **Balanced Diet:** Eat a healthy diet and control portion sizes.\n   * **Regular Exercise:** Aim for at least 30 minutes of moderate-intensity exercise most days of the week.\n\n4. **Don't Smoke:** Smoking damages your blood vessels and increases your risk of CAD.\n\n5. **Regular Checkups:** See your doctor for regular checkups and blood pressure and cholesterol screenings.\n\n**Important Note:** This information is for general knowledge and should not be considered medical advice. It's essential to consult with your doctor for personalized guidance and treatment recommendations.\n"

"Let's break down these medical terms and then talk about what you can do to manage your heart health.\n\n**Understanding the Terms**\n\n* **Age:** Your age is 24, which is generally considered young and puts you at lower risk for coronary artery disease (CAD).\n* **Sex:** Being male (sex = 1) slightly increases your risk of CAD compared to females.\n* **Chest Pain Type:** 0 indicates you didn't report experiencing chest pain at rest.\n* **Resting Systolic BP:** 120 mmHg is a normal blood pressure reading.\n* **Cholesterol:** 161 mg/dL is considered high cholesterol. High cholesterol is a major risk factor for CAD.\n* **Fasting Blood Sugar Flag:** 0 means your fasting blood sugar level is normal.\n* **Resting ECG:** 0 indicates your resting electrocardiogram (ECG) was normal.\n* **Max Heart Rate:** 180 bpm is a good maximum heart rate for your age. It suggests good cardiovascular fitness.\n* **Exercise-Induced Angina:** 0 means you didn't experience chest pain during the exercise test.